In [ ]:
import pandas as pd
from google.colab import files

# 1. This will prompt you to select your 'hospital_readmissions.csv' from your computer
uploaded = files.upload()

# 2. Load the data
df = pd.read_csv('hospital_readmissions.csv')

# 3. Create our new features (re-doing your SQL logic)
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == 'yes' else 0)

def categorize_stay(days):
    if days <= 3: return 'Short Stay'
    elif days <= 7: return 'Medium Stay'
    else: return 'Long Stay'

df['stay_category'] = df['time_in_hospital'].apply(categorize_stay)

# 4. Show a quick preview to make sure it worked
print("Data Loaded Successfully!")
df.head()

# 5. Export the processed data back to a CSV for Power BI
df.to_csv('processed_hospital_data.csv', index=False)
files.download('processed_hospital_data.csv')



Saving hospital_readmissions.csv to hospital_readmissions (1).csv
Data Loaded Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# ---------------------------------------
# 1. Load Processed Dataset
# ---------------------------------------
df = pd.read_csv("processed_hospital_data.csv")

# ---------------------------------------
# 2. Feature Selection
# ---------------------------------------
categorical_cols = [
    "age",
    "medical_specialty",
    "diag_1",
    "glucose_test",
    "A1Ctest",
    "change",
    "diabetes_med",
    "stay_category"
]

numeric_cols = [
    "time_in_hospital",
    "n_lab_procedures",
    "n_procedures",
    "n_medications",
    "n_outpatient",
    "n_inpatient",
    "n_emergency"
]

# Encode categorical features
X_categorical = pd.get_dummies(df[categorical_cols], drop_first=True)

# Numeric features
X_numeric = df[numeric_cols]

# Combine features
X = pd.concat([X_numeric, X_categorical], axis=1)

# Target
y = df["readmitted_binary"]

# ---------------------------------------
# 3. Train-Test Split
# ---------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------------------
# 4. Train Model
# ---------------------------------------
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

# ---------------------------------------
# 5. Predictions
# ---------------------------------------
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

# ---------------------------------------
# 6. Model Metrics
# ---------------------------------------
accuracy = accuracy_score(y_test, predictions)
roc_score = roc_auc_score(y_test, probabilities)

print("Model Accuracy:", round(accuracy, 4))
print("ROC-AUC Score:", round(roc_score, 4))

# ---------------------------------------
# 7. Create Dataset for Power BI
# ---------------------------------------
results = X_test.copy()
results["actual_readmission"] = y_test.values
results["predicted_readmission"] = predictions
results["predicted_probability"] = probabilities

# Save file for Power BI
results.to_csv("powerbi_readmission_predictions.csv", index=False)

print("Dataset saved: powerbi_readmission_predictions.csv")

Model Accuracy: 0.6244
ROC-AUC Score: 0.6557
Dataset saved: powerbi_readmission_predictions.csv
